# Ddri 2차 군집화 추가 피처 생성 노트북

목적:
- 2차 군집화 후보 피처 6개를 별도 CSV로 생성한다.
- 기존 군집화 feature CSV에 바로 합치지 않고, 팀 검토용 독립 산출물로 관리한다.
- baseline 전처리 기준과 동일한 cleaning 로직을 재사용해 기준이 분기되지 않도록 한다.

이 노트북에서 생성하는 피처:
- `same_station_return_ratio` : 동일 대여소 반납 비율
- `net_flow_mean` : 평균 순유출입
- `abs_net_flow_mean` : 평균 절대 순유출입
- `morning_peak_ratio` : 아침 출근 시간대 비율
- `evening_peak_ratio` : 저녁 퇴근 시간대 비율
- `lunch_ratio` : 점심 시간대 비율

## 입력과 출력

입력:
- 원천 따릉이 이용 로그 CSV
- baseline 스크립트의 공통 대여소/유효 반납 대여소 기준

출력:
- `cheng80/ddri_station_cluster_additional_features_train_2023_2024.csv`
- `cheng80/ddri_station_cluster_additional_features_test_2025.csv`

주의:
- 기존 `ddri_station_cluster_features_train_with_labels.csv` 같은 파일에는 합치지 않는다.
- 이 노트북은 검토용 추가 피처 CSV만 생성한다.

## 전처리 기준

이 노트북은 `works/01_clustering/01_baseline/ddri_station_clustering_baseline.py`의 함수를 그대로 가져와 사용한다.
즉 아래 기준이 동일하게 적용된다.

- 필수 컬럼 결측 제거
- `이용시간(분) <= 0` 제거
- `이용거리(M) <= 0` 제거
- `동일 대여소 반납 + 5분 이하` 제거
- 공통 대여소 기준 밖 대여 제거
- 해당 연도 강남구 기준 밖 반납 제거

이 기준을 baseline과 맞춰야, 나중에 추가 피처를 합칠 때 데이터셋 정합성이 유지된다.

In [1]:
from pathlib import Path
import sys

import pandas as pd

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
BASELINE_DIR = BASE_DIR / 'works' / '01_clustering' / '01_baseline'
OUTPUT_DATA_DIR = BASE_DIR / 'cheng80'

if str(BASELINE_DIR) not in sys.path:
    sys.path.append(str(BASELINE_DIR))

from ddri_station_clustering_baseline import file_groups, load_clean_group, load_station_frames

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
BASE_DIR, OUTPUT_DATA_DIR

(PosixPath('/Users/cheng80/Desktop/ddri_work'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/cheng80'))

## 이벤트 단위 데이터를 station-day 운영 지표로 바꾸는 함수

먼저 이벤트 로그를 `station-day` 수준으로 바꿔야 순유출입과 self-return 비율을 계산할 수 있다.

생성 중간 지표:
- `rental_count`
- `return_count`
- `same_station_return_count`
- `net_flow = rental_count - return_count`

In [2]:
def build_station_day_metrics(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work['date'] = work['대여일시'].dt.normalize()
    work['return_date'] = work['반납일시'].dt.normalize()

    rental_df = (
        work.groupby(['대여 대여소번호', 'date'])
        .size()
        .reset_index(name='rental_count')
        .rename(columns={'대여 대여소번호': 'station_id'})
    )
    return_df = (
        work.groupby(['반납대여소번호', 'return_date'])
        .size()
        .reset_index(name='return_count')
        .rename(columns={'반납대여소번호': 'station_id', 'return_date': 'date'})
    )
    same_station_df = work[
        (work['대여 대여소번호'] == work['반납대여소번호'])
        & (work['date'] == work['return_date'])
    ]
    same_station_df = (
        same_station_df.groupby(['대여 대여소번호', 'date'])
        .size()
        .reset_index(name='same_station_return_count')
        .rename(columns={'대여 대여소번호': 'station_id'})
    )

    station_day = (
        rental_df.merge(return_df, on=['station_id', 'date'], how='left')
        .merge(same_station_df, on=['station_id', 'date'], how='left')
        .sort_values(['station_id', 'date'])
        .reset_index(drop=True)
    )
    station_day['return_count'] = station_day['return_count'].fillna(0).astype(int)
    station_day['same_station_return_count'] = station_day['same_station_return_count'].fillna(0).astype(int)
    station_day['net_flow'] = station_day['rental_count'] - station_day['return_count']
    return station_day


## 시간대 비율 피처 함수

이 단계에서는 이벤트 단위의 `대여일시.hour`를 사용한다.

시간대 정의:
- 아침 출근: `07, 08, 09`
- 저녁 퇴근: `18, 19, 20`
- 점심: `11, 12, 13`

각 지표는 `해당 시간대 대여 수 / 전체 대여 수` 비율이다.

In [3]:
def build_time_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work['hour'] = work['대여일시'].dt.hour

    total = work.groupby('대여 대여소번호').size()
    morning = work.loc[work['hour'].isin([7, 8, 9])].groupby('대여 대여소번호').size()
    evening = work.loc[work['hour'].isin([18, 19, 20])].groupby('대여 대여소번호').size()
    lunch = work.loc[work['hour'].isin([11, 12, 13])].groupby('대여 대여소번호').size()

    ratio_df = pd.DataFrame({'station_id': total.index})
    ratio_df['morning_peak_ratio'] = (morning / total).reindex(total.index).fillna(0.0).values
    ratio_df['evening_peak_ratio'] = (evening / total).reindex(total.index).fillna(0.0).values
    ratio_df['lunch_ratio'] = (lunch / total).reindex(total.index).fillna(0.0).values
    return ratio_df.sort_values('station_id').reset_index(drop=True)


## station 수준 최종 추가 피처 테이블 생성 함수

이 함수는 `station-day` 운영 지표와 시간대 비율을 합쳐서,
검토용 `station` 수준 추가 피처 테이블을 만든다.

핵심 계산식:
- `same_station_return_ratio = sum(same_station_return_count) / sum(rental_count)`
- `net_flow_mean = mean(net_flow)`
- `abs_net_flow_mean = mean(abs(net_flow))`

In [4]:
def build_additional_features(df: pd.DataFrame) -> pd.DataFrame:
    station_day = build_station_day_metrics(df)

    station_level = (
        station_day.groupby('station_id')
        .agg(
            rental_count_sum=('rental_count', 'sum'),
            same_station_return_count_sum=('same_station_return_count', 'sum'),
            net_flow_mean=('net_flow', 'mean'),
            abs_net_flow_mean=('net_flow', lambda s: s.abs().mean()),
        )
        .reset_index()
    )
    station_level['same_station_return_ratio'] = (
        station_level['same_station_return_count_sum'] / station_level['rental_count_sum']
    ).fillna(0.0)

    station_level = station_level.drop(columns=['rental_count_sum', 'same_station_return_count_sum'])
    time_ratio_df = build_time_ratio_features(df)

    feature_df = (
        station_level.merge(time_ratio_df, on='station_id', how='inner')
        .sort_values('station_id')
        .reset_index(drop=True)
    )

    return feature_df[
        [
            'station_id',
            'same_station_return_ratio',
            'net_flow_mean',
            'abs_net_flow_mean',
            'morning_peak_ratio',
            'evening_peak_ratio',
            'lunch_ratio',
        ]
    ]


## 데이터 로드 및 정제

이 셀은 baseline에서 쓰는 cleaning 함수를 그대로 호출한다.
학습용은 `2023 + 2024`, 테스트용은 `2025`로 분리한다.

In [5]:
station_id_sets, common_station_ids = load_station_frames()
common_station_id_set = set(common_station_ids)
rental_groups = file_groups()

train_df_2023, _ = load_clean_group(
    rental_groups[2023], station_id_sets[2023], common_station_id_set, 'train_2023'
)
train_df_2024, _ = load_clean_group(
    rental_groups[2024], station_id_sets[2024], common_station_id_set, 'train_2024'
)
test_df_2025, _ = load_clean_group(
    rental_groups[2025], station_id_sets[2025], common_station_id_set, 'test_2025'
)

train_df = pd.concat([train_df_2023, train_df_2024], ignore_index=True)

print('train_event_rows =', len(train_df))
print('test_event_rows =', len(test_df_2025))

[train_2023] 1/12 2301_강남구_따릉이_이용정보.csv: 34,937 -> 30,957
[train_2023] 2/12 2302_강남구_따릉이_이용정보.csv: 49,372 -> 44,188
[train_2023] 3/12 2303_강남구_따릉이_이용정보.csv: 80,572 -> 72,079


[train_2023] 4/12 2304_강남구_따릉이_이용정보.csv: 91,717 -> 81,967
[train_2023] 5/12 2305_강남구_따릉이_이용정보.csv: 107,880 -> 97,042


[train_2023] 6/12 2306_강남구_따릉이_이용정보.csv: 105,610 -> 95,109
[train_2023] 7/12 2307_강남구_따릉이_이용정보.csv: 85,691 -> 76,616


[train_2023] 8/12 2308_강남구_따릉이_이용정보.csv: 92,398 -> 82,801
[train_2023] 9/12 2309_강남구_따릉이_이용정보.csv: 105,436 -> 93,874


[train_2023] 10/12 2310_강남구_따릉이_이용정보.csv: 124,747 -> 111,821
[train_2023] 11/12 2311_강남구_따릉이_이용정보.csv: 85,518 -> 77,369
[train_2023] 12/12 2312_강남구_따릉이_이용정보.csv: 48,602 -> 43,819


...

[train_2024] 10/12 2410_강남구_따릉이_이용정보.csv: 101,558 -> 95,690
[train_2024] 11/12 2411_강남구_따릉이_이용정보.csv: 78,883 -> 73,991
[train_2024] 12/12 2412_강남구_따릉이_이용정보.csv: 48,645 -> 45,507


[test_2025] 1/12 2501_강남구_따릉이_이용정보.csv: 37,502 -> 35,410
[test_2025] 2/12 2502_강남구_따릉이_이용정보.csv: 36,065 -> 34,375
[test_2025] 3/12 2503_강남구_따릉이_이용정보.csv: 63,383 -> 61,268


[test_2025] 4/12 2504_강남구_따릉이_이용정보.csv: 85,989 -> 83,339
[test_2025] 5/12 2505_강남구_따릉이_이용정보.csv: 85,055 -> 82,574
[test_2025] 6/12 2506_강남구_따릉이_이용정보.csv: 91,073 -> 88,635


[test_2025] 7/12 2507_강남구_따릉이_이용정보.csv: 84,314 -> 79,542
[test_2025] 8/12 2508_강남구_따릉이_이용정보.csv: 86,307 -> 79,840
[test_2025] 9/12 2509_강남구_따릉이_이용정보.csv: 94,073 -> 86,051


[test_2025] 10/12 2510_강남구_따릉이_이용정보.csv: 82,459 -> 75,288
[test_2025] 11/12 2511_강남구_따릉이_이용정보.csv: 77,918 -> 71,552
[test_2025] 12/12 2512_강남구_따릉이_이용정보.csv: 45,259 -> 42,015
train_event_rows = 1844049
test_event_rows = 819889


## 추가 피처 생성 및 미리보기

생성 후에는 우선 값 분포와 row 수가 예상과 맞는지만 확인한다.
기존 군집 feature CSV와는 아직 합치지 않는다.

In [6]:
train_feature_df = build_additional_features(train_df)
test_feature_df = build_additional_features(test_df_2025)

print('train_feature_rows =', len(train_feature_df))
print('test_feature_rows =', len(test_feature_df))

train_feature_df.head()

train_feature_rows = 164
test_feature_rows = 162


,station_id,same_station_return_ratio,net_flow_mean,abs_net_flow_mean,morning_peak_ratio,evening_peak_ratio,lunch_ratio
0,2301,0.411458,-5.845404,6.928969,0.073346,0.281066,0.099939
1,2302,0.110281,-2.447887,4.833803,0.122123,0.216564,0.118866
2,2303,0.066614,2.009576,4.854993,0.187854,0.206286,0.144154
3,2304,0.034908,4.837916,5.246020,0.149726,0.261123,0.127310
4,2305,0.052192,-0.163028,2.491994,0.089908,0.319062,0.160041


In [7]:
train_feature_df.describe().round(4)

,station_id,same_station_return_ratio,net_flow_mean,abs_net_flow_mean,morning_peak_ratio,evening_peak_ratio,lunch_ratio
count,164.0000,164.0000,164.0000,164.0000,164.0000,164.0000,164.0000
mean,2968.0000,0.0883,0.3888,4.1352,0.1680,0.2074,0.1413
std,919.5597,0.0565,3.6200,2.3285,0.0675,0.0403,0.0285
min,2301.0000,0.0154,-11.6361,1.0302,0.0457,0.0772,0.0803
25%,2347.7500,0.0523,-1.4465,2.7661,0.1211,0.1818,0.1212
50%,2399.5000,0.0790,0.3405,3.4855,0.1590,0.2068,0.1402
75%,3623.2500,0.1113,1.6350,4.7642,0.2034,0.2295,0.1561
max,4930.0000,0.4115,16.2893,16.5234,0.4081,0.3778,0.2745


## CSV 저장

이 셀을 실행하면 검토용 추가 피처 CSV가 저장된다.

저장 파일:
- 학습용: `ddri_station_cluster_additional_features_train_2023_2024.csv`
- 테스트용: `ddri_station_cluster_additional_features_test_2025.csv`

In [8]:
train_output_path = OUTPUT_DATA_DIR / 'ddri_station_cluster_additional_features_train_2023_2024.csv'
test_output_path = OUTPUT_DATA_DIR / 'ddri_station_cluster_additional_features_test_2025.csv'

train_feature_df.to_csv(train_output_path, index=False)
test_feature_df.to_csv(test_output_path, index=False)

print('saved:', train_output_path)
print('saved:', test_output_path)

saved: /Users/cheng80/Desktop/ddri_work/cheng80/ddri_station_cluster_additional_features_train_2023_2024.csv
saved: /Users/cheng80/Desktop/ddri_work/cheng80/ddri_station_cluster_additional_features_test_2025.csv


## 최종 메모

- 이 노트북은 별도 검토용 피처 CSV 생성이 목적이다.
- 팀 검토가 끝나기 전까지 기존 군집 feature CSV에는 병합하지 않는다.
- 이후 합칠 때는 `station_id` 기준 join을 사용하면 된다.